# This is your Python notebook
Start solving by adding a new cell below — write and run code, or jot notes in text cells as you go.

In [94]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, f_oneway, chi2_contingency, shapiro, levene


In [95]:
df = pd.read_csv("bike_sharing.csv")

# Basic info and shape
print(f"Dataset Shape: {df.shape}")
print(df.head())
print(df.info())
print(df.tail())

### Problem Statement & Business Objective

**Business Objective:**
Yulu aims to identify the specific factors that influence the demand for shared electric cycles. Following a recent dip in revenue, the company needs a data-driven approach to understand which variables significantly impact usage patterns to optimize operations and reverse the revenue trend.

**Problem Statement:**
To determine the statistical significance of various environmental and temporal factors on the demand for shared electric cycles. Specifically, Yulu wants to answer:
*   Which variables (e.g., weather, season, working days) are significant predictors of cycle demand?
*   How well do these variables explain the fluctuations in the number of rentals?

**Key Factors Influencing Demand:**
*   **Temporal Factors:** Seasons (Spring, Summer, Fall, Winter), Holidays, and Working Days vs. Weekends.
*   **Weather Conditions:** Temperature, 'Feels-like' temperature (atemp), Humidity, and Wind speed.
*   **Weather Categories:** Clear/Cloudy vs. Rain/Snow/Storm conditions.

In [96]:
# Check for missing values
print("Missing Values:\n", df.isnull().sum())

# Convert datetime
df['datetime'] = pd.to_datetime(df['datetime'])

# Convert categorical variables to 'category' type for better analysis
cat_cols = ['season', 'holiday', 'workingday', 'weather']
for col in cat_cols:
    df[col] = df[col].astype('category')

display(df.describe())

In [97]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_count}")

In [98]:
# Identify numerical columns
num_cols = ['temp', 'atemp', 'humidity', 'windspeed', 'casual', 'registered', 'count']

# 1. Visual Outlier Detection using Boxplots
plt.figure(figsize=(15, 10))
for i, col in enumerate(num_cols):
    plt.subplot(3, 3, i+1)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()

# 2. Outlier Detection using IQR Method
def detect_outliers_iqr(data):
    outliers_report = {}
    for col in num_cols:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
        outliers_report[col] = len(outliers)
    return outliers_report

outlier_counts = detect_outliers_iqr(df)
print("Number of outliers detected per variable using IQR method:")
for col, count in outlier_counts.items():
    print(f"{col}: {count} outliers")

In [99]:
# Non-Graphical Analysis: Unique values and Value Counts for categorical variables
cat_cols = ['season', 'holiday', 'workingday', 'weather']

for col in cat_cols:
    print(f"--- Analysis for '{col}' ---")
    print(f"Unique values: {df[col].unique().tolist()}")
    print(f"Value counts:\n{df[col].value_counts()}")
    print(f"Percentage distribution:\n{df[col].value_counts(normalize=True) * 100}")
    print("\n")

In [100]:
# Univariate Analysis: Distribution of continuous variables
continuous_vars = ['temp', 'atemp', 'humidity', 'windspeed', 'count']

plt.figure(figsize=(15, 12))
for i, col in enumerate(continuous_vars):
    plt.subplot(3, 2, i + 1)
    sns.histplot(df[col], kde=True, color='skyblue')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [101]:
# Univariate Analysis: Frequency distribution of categorical variables
cat_vars = ['season', 'weather', 'holiday', 'workingday']

plt.figure(figsize=(16, 10))
for i, col in enumerate(cat_vars):
    plt.subplot(2, 2, i + 1)
    sns.countplot(data=df, x=col, hue=col, palette='viridis', legend=False)
    plt.title(f'Count Plot of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')

plt.tight_layout()
plt.show()

In [102]:
# Bivariate Analysis: Demand Patterns across Categorical Variables
analysis_cols = ['season', 'weather', 'workingday', 'holiday']

# Visualizing patterns using boxplots
plt.figure(figsize=(18, 12))
for i, col in enumerate(analysis_cols):
    plt.subplot(2, 2, i + 1)
    sns.boxplot(data=df, x=col, y='count', hue=col, palette='Set2', legend=False)
    plt.title(f'Rental Demand by {col.capitalize()}')
    plt.xlabel(col.capitalize())
    plt.ylabel('Total Rental Count')

plt.tight_layout()
plt.show()

# Documenting patterns with statistical summaries
print("--- Summary Statistics for Demand --- ")
for col in analysis_cols:
    summary = df.groupby(col)['count'].describe()[['mean', '50%', 'std']]
    print(f"\nDemand patterns by {col.capitalize()}:\n{summary}")

In [103]:
# Bivariate Analysis: Dependent variable (count) vs Independent Variables

# 1. Numerical vs Numerical (Scatter plots to check for correlation)
num_vars = ['temp', 'atemp', 'humidity', 'windspeed']
plt.figure(figsize=(15, 10))
for i, col in enumerate(num_vars):
    plt.subplot(2, 2, i + 1)
    sns.scatterplot(data=df, x=col, y='count', alpha=0.2)
    plt.title(f'Count vs {col.capitalize()}')
plt.tight_layout()
plt.show()

# 2. Categorical vs Numerical (Boxplots to compare distributions)
plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
sns.boxplot(data=df, x='season', y='count', hue='season', palette='coolwarm', legend=False)
plt.title('Demand across Seasons')

plt.subplot(1, 3, 2)
sns.boxplot(data=df, x='weather', y='count', hue='weather', palette='magma', legend=False)
plt.title('Demand across Weather Conditions')

plt.subplot(1, 3, 3)
sns.boxplot(data=df, x='workingday', y='count', hue='workingday', palette='Set1', legend=False)
plt.title('Demand: Working Day vs Weekend')

plt.tight_layout()
plt.show()

### Key Observations from Bivariate Analysis

Based on the visualizations and statistical summaries, we can identify several key patterns in Yulu's bike rental demand:

1.  **Seasonal Impact:**
    *   **Fall (Season 3)** and **Summer (Season 2)** see the highest average demand (~234 and ~215 respectively).
    *   **Spring (Season 1)** has the lowest demand by a significant margin (~116), nearly half of the peak seasonal demand.

2.  **Weather Conditions:**
    *   **Clear/Partly Cloudy (Weather 1)** conditions drive the most rentals.
    *   Demand drops significantly during **Light Rain/Snow (Weather 3)**.
    *   Heavy rain/storm conditions (Weather 4) are extremely rare in this dataset but show minimal usage, suggesting weather is a critical predictor.

3.  **Working Day vs. Weekend:**
    *   The **median demand** is higher on working days compared to weekends/holidays.
    *   However, the **mean demand** is quite similar (Working Day: ~193 vs Non-Working Day: ~188), suggesting that while the volume is consistent, the usage patterns (likely commuting vs. leisure) might differ significantly.

4.  **Environmental Correlations:**
    *   **Temperature & Feeling Temperature (atemp):** Show a positive correlation with demand; as it gets warmer, rentals increase until they plateau at very high temperatures.
    *   **Humidity:** Shows a negative relationship; high humidity levels (above 60-70%) appear to suppress demand.

In [104]:
plt.figure(figsize=(14, 6))

# 1. Boxplot to see distribution, medians, and outliers
plt.subplot(1, 2, 1)
sns.boxplot(data=df, x='workingday', y='count', hue='workingday', palette='muted', legend=False)
plt.title('Distribution of Rental Count: Working Day vs Non-Working Day')
plt.xlabel('Working Day (0=No, 1=Yes)')
plt.ylabel('Total Rental Count')

# 2. Violin plot to see the density and distribution shape
plt.subplot(1, 2, 2)
sns.violinplot(data=df, x='workingday', y='count', hue='workingday', palette='muted', inner='quartile', legend=False)
plt.title('Density of Rental Count: Working Day vs Non-Working Day')
plt.xlabel('Working Day (0=No, 1=Yes)')
plt.ylabel('Total Rental Count')

plt.tight_layout()
plt.show()

### Hypothesis Testing: 2-Sample T-Test (Working Day vs. Demand)

We will conduct a 2-sample independent T-test to determine if there is a significant difference in the mean number of bike rentals between working days and non-working days.

*   **Null Hypothesis ($H_0$):** The mean number of bike rentals is the same for working days and non-working days. ($\mu_1 = \mu_2$)
*   **Alternative Hypothesis ($H_1$):** The mean number of bike rentals is significantly different for working days and non-working days. ($\mu_1 \neq \mu_2$)

**Significance Level ($\alpha$):** 0.05

### Statistical Test Selection and Justification

**Selected Test:** 2-Sample Independent T-Test (Unpaired T-test).

**Justification:**
1.  **Nature of the Groups:** We are comparing two independent groups: 'Working Day' (1) and 'Non-Working Day' (0). A rental occurring on a working day has no dependency on a rental occurring on a holiday/weekend.
2.  **Comparison of Means:** The objective is to determine if the average (mean) demand differs between these two distinct categories.
3.  **Data Type:** The independent variable (`workingday`) is categorical with exactly two levels, and the dependent variable (`count`) is continuous.

**Assumptions to Verify:**
*   **Independence:** Observations are independent (satisfied by the data collection process).
*   **Normality:** The target variable `count` should follow a normal distribution (we will check this using the Shapiro-Wilk test or visual QQ-plots).
*   **Equal Variance (Homoscedasticity):** The variance of demand should be approximately equal across both groups (we will check this using Levene’s test).

In [105]:
import scipy.stats as stats

# Splitting the demand data into two groups
working_day_count = df[df['workingday'] == 1]['count']
non_working_day_count = df[df['workingday'] == 0]['count']

# 1. Normality Test (Shapiro-Wilk)
# Note: Shapiro-Wilk can be sensitive to large N, so we take a sample of 5000 if necessary
shapiro_working = stats.shapiro(working_day_count.sample(min(5000, len(working_day_count)), random_state=42))
shapiro_non_working = stats.shapiro(non_working_day_count.sample(min(5000, len(non_working_day_count)), random_state=42))

print("--- Normality Tests (Shapiro-Wilk) ---")
print(f"Working Day p-value: {shapiro_working.pvalue:.4f}")
print(f"Non-Working Day p-value: {shapiro_non_working.pvalue:.4f}")

# 2. Visual Normality Check (QQ-Plots)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
stats.probplot(working_day_count, dist='norm', plot=plt)
plt.title('QQ-Plot: Working Day')

plt.subplot(1, 2, 2)
stats.probplot(non_working_day_count, dist='norm', plot=plt)
plt.title('QQ-Plot: Non-Working Day')
plt.show()

# 3. Equality of Variances (Levene's Test)
levene_stat, levene_p = stats.levene(working_day_count, non_working_day_count)
print("\n--- Equality of Variance (Levene's Test) ---")
print(f"Levene p-value: {levene_p:.4f}")

# 4. Sample Size Adequacy
print("\n--- Sample Size Adequacy ---")
print(f"N (Working Day): {len(working_day_count)}")
print(f"N (Non-Working Day): {len(non_working_day_count)}")
print("Comment: With N > 30 for both groups, the Central Limit Theorem suggests the t-test is relatively robust to normality violations.")

In [106]:
# Perform 2-sample t-test
# Based on Levene's test (p-value = 0.9438), we assume equal variances
t_stat, p_value = stats.ttest_ind(working_day_count, non_working_day_count, equal_var=True)

print("--- 2-Sample T-Test (Working Day vs. Demand) ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# Inference
alpha = 0.05
if p_value < alpha:
    print("\nConclusion: Since p-value < 0.05, we reject the Null Hypothesis (H0).")
    print("There is a statistically significant difference in bike rental demand between working and non-working days.")
else:
    print("\nConclusion: Since p-value >= 0.05, we fail to reject the Null Hypothesis (H0).")
    print("There is no statistically significant difference in bike rental demand between working and non-working days.")

In [107]:
# 1. Normality Test (Shapiro-Wilk) for each weather category
# Note: Weather 4 has only 1 observation, so it cannot be tested for normality or variance.
weather_groups = [df[df['weather'] == i]['count'] for i in [1, 2, 3]]
weather_names = ['Clear', 'Mist', 'Light Rain']

print("--- Normality Tests (Shapiro-Wilk) ---")
for name, group in zip(weather_names, weather_groups):
    # Sampling 5000 if necessary due to Shapiro-Wilk constraints
    stat, p = stats.shapiro(group.sample(min(5000, len(group)), random_state=42))
    print(f"{name} Weather: p-value = {p:.4f}")

# 2. Equality of Variances (Levene's Test)
# We test groups 1, 2, and 3 (Weather 4 is excluded due to single data point)
levene_stat, levene_p = stats.levene(*weather_groups)

print("\n--- Equality of Variance (Levene's Test) ---")
print(f"Levene p-value: {levene_p:.4e}")

# 3. Visual Normality Check (QQ-Plots)
plt.figure(figsize=(18, 5))
for i, (name, group) in enumerate(zip(weather_names, weather_groups)):
    plt.subplot(1, 3, i + 1)
    stats.probplot(group, dist='norm', plot=plt)
    plt.title(f'QQ-Plot: {name}')

plt.tight_layout()
plt.show()

In [108]:
# Preparing groups for ANOVA
g1 = df[df['weather'] == 1]['count']
g2 = df[df['weather'] == 2]['count']
g3 = df[df['weather'] == 3]['count']
g4 = df[df['weather'] == 4]['count']

# Performing One-Way ANOVA
f_stat, p_val = stats.f_oneway(g1, g2, g3, g4)

print("--- One-Way ANOVA Result (Weather vs. Demand) ---")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_val:.4e}")

# Interpretation
alpha = 0.05
if p_val < alpha:
    print(f"\nConclusion: Since p-value ({p_val:.4e}) < {alpha}, we reject the Null Hypothesis (H0).")
    print("There is a statistically significant difference in the number of cycles rented across different weather conditions.")
else:
    print(f"\nConclusion: Since p-value ({p_val:.4f}) >= {alpha}, we fail to reject the Null Hypothesis (H0).")
    print("There is no statistically significant difference in the number of cycles rented across different weather conditions.")

In [109]:
# Visual analysis of count across season categories
plt.figure(figsize=(16, 6))

# Mapping season numbers to names for readability
season_map = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}

# 1. Boxplot: Distribution and Outliers across Seasons
plt.subplot(1, 2, 1)
sns.boxplot(data=df, x='season', y='count', hue='season', palette='coolwarm', legend=False)
plt.xticks(ticks=[0, 1, 2, 3], labels=['Spring', 'Summer', 'Fall', 'Winter'])
plt.title('Rental Count Distribution by Season')
plt.xlabel('Season')
plt.ylabel('Total Rental Count')

# 2. Violin Plot: Demand Density across Seasons
plt.subplot(1, 2, 2)
sns.violinplot(data=df, x='season', y='count', hue='season', palette='coolwarm', inner='quartile', legend=False)
plt.xticks(ticks=[0, 1, 2, 3], labels=['Spring', 'Summer', 'Fall', 'Winter'])
plt.title('Demand Density by Season')
plt.xlabel('Season')
plt.ylabel('Total Rental Count')

plt.tight_layout()
plt.show()

Hypothesis Testing: ANOVA (Season vs. Demand)
To determine if the number of bike rentals differs significantly across the four seasons, we will perform a One-Way ANOVA.

Null Hypothesis ( H0 ): The mean number of bike rentals is the same across all four seasons (Spring, Summer, Fall, and Winter).
μ1=μ2=μ3=μ4 
Alternative Hypothesis ( H1 ): At least one season has a significantly different mean number of bike rentals compared to the others.
Significance Level ( α ): 0.05

### Statistical Test Selection and Justification (Season Analysis)

**Selected Test:** One-Way ANOVA (Analysis of Variance).

**Justification:**
1. **Number of Groups:** We are comparing the means of **four independent groups** (Spring, Summer, Fall, and Winter). While a t-test is limited to two groups, ANOVA is designed to handle three or more groups.
2. **Type of Variables:** The independent variable (`season`) is categorical with four levels, and the dependent variable (`count`) is continuous numerical data.
3. **Efficiency:** Performing multiple t-tests (e.g., Spring vs. Summer, Summer vs. Fall, etc.) increases the probability of committing a Type I error (false positive). ANOVA controls the overall error rate by testing the global hypothesis that all means are equal in a single step.

**Assumptions to be Checked:**
* **Normality:** The distribution of the residuals or the data within each group should be approximately normal.
* **Homogeneity of Variance:** The variance among the groups should be approximately equal (checked via Levene's test).
* **Independence:** Observations in each group are independent of each other.

In [110]:
# 1. Normality Test (Shapiro-Wilk) for each Season category
season_groups = [df[df['season'] == i]['count'] for i in [1, 2, 3, 4]]
season_names = ['Spring', 'Summer', 'Fall', 'Winter']

print("--- Normality Tests (Shapiro-Wilk) ---")
for name, group in zip(season_names, season_groups):
    # Sampling 5000 if necessary due to Shapiro-Wilk constraints
    stat, p = stats.shapiro(group.sample(min(5000, len(group)), random_state=42))
    print(f"{name} Season: p-value = {p:.4f}")

# 2. Equality of Variances (Levene's Test) across all 4 seasons
levene_stat, levene_p = stats.levene(*season_groups)

print("\n--- Equality of Variance (Levene's Test) ---")
print(f"Levene p-value: {levene_p:.4e}")

# 3. Visual Normality Check (QQ-Plots)
plt.figure(figsize=(20, 5))
for i, (name, group) in enumerate(zip(season_names, season_groups)):
    plt.subplot(1, 4, i + 1)
    stats.probplot(group, dist='norm', plot=plt)
    plt.title(f'QQ-Plot: {name}')

plt.tight_layout()
plt.show()

In [111]:
# Preparing groups for ANOVA (Season vs. Demand)
g1 = df[df['season'] == 1]['count']
g2 = df[df['season'] == 2]['count']
g3 = df[df['season'] == 3]['count']
g4 = df[df['season'] == 4]['count']

# Performing One-Way ANOVA
f_stat, p_val = stats.f_oneway(g1, g2, g3, g4)

print("--- One-Way ANOVA Result (Season vs. Demand) ---")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_val:.4e}")

# Interpretation
alpha = 0.05
if p_val < alpha:
    print(f"\nConclusion: Since p-value ({p_val:.4e}) < {alpha}, we reject the Null Hypothesis (H0).")
    print("There is a statistically significant difference in the number of cycles rented across different seasons.")
else:
    print(f"\nConclusion: Since p-value ({p_val:.4e}) >= {alpha}, we fail to reject the Null Hypothesis (H0).")
    print("There is no statistically significant difference in the number of cycles rented across different seasons.")

In [113]:
# Normalized crosstab for proportion visualization
contingency_pct = pd.crosstab(
    df["season"], df["weather"], normalize="index"
) * 100

contingency_pct.plot(kind="bar", stacked=True, figsize=(9, 6), colormap="viridis")
plt.title("Proportional Distribution of Weather across Seasons")
plt.xlabel("Season (1: Spring, 2: Summer, 3: Fall, 4: Winter)")
plt.ylabel("Percentage (%)")
plt.legend(title="Weather", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Null Hypothesis ($\text{H}_0$): Weather conditions and seasons are independent of each other. (There is no association between weather and season.)
Alternative Hypothesis ($\text{H}_1$): Weather conditions and seasons are dependent on each other. (There is a significant association between weather and season.)

The appropriate statistical test is the Chi-Square Test of Independence.
Justification
Categorical Nature of Both Variables: Both season (e.g., Spring, Summer, Fall, Winter) and weather (e.g., Clear, Mist, Light Rain, Heavy Rain) are categorical variables. The Chi-Square test is specifically designed to handle frequency/count data formed by two categorical variables.

In [114]:
# 1. Create the contingency table (Observed Frequencies)
contingency_tab = pd.crosstab(df["season"], df["weather"])

# 2. Perform Chi-Square Test to obtain Expected Frequencies
chi2_stat, p_val, dof, expected_freq = chi2_contingency(contingency_tab)

# 3. Convert expected frequencies to a DataFrame for clear visual comparison
expected_df = pd.DataFrame(
    expected_freq, index=contingency_tab.index, columns=contingency_tab.columns
)

print("--- Observed Frequencies ---")
print(contingency_tab)

print("\n--- Expected Frequencies ---")
print(expected_df.round(2))

# 4. Check if any expected frequencies are less than 5
cells_below_5 = (expected_df < 5).sum().sum()
print(f"\nNumber of cells with expected frequency < 5: {cells_below_5}")

In [116]:
# 1. Create a contingency table (cross-tabulation)
contingency_table = pd.crosstab(df['season'], df['weather'])
print("--- Contingency Table (Season vs. Weather) ---")
display(contingency_table)

# 2. Perform the Chi-Square Test of Independence
chi2_stat, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\nChi-Square Statistic: {chi2_stat:.4f}")
print(f"P-value: {p_val_chi2:.4e}")
print(f"Degrees of Freedom: {dof}")

# 3. Interpretation
alpha = 0.05
if p_val_chi2 < alpha:
    print(f"\nConclusion: Since p-value ({p_val_chi2:.4e}) < {alpha}, we reject the Null Hypothesis (H0).")
    print("Weather is significantly dependent on the season.")
else:
    print(f"\nConclusion: Since p-value ({p_val_chi2:.4f}) >= {alpha}, we fail to reject the Null Hypothesis (H0).")
    print("There is no significant dependency between weather and season.")

# 
#  CONSOLIDATED KEY INSIGHTS & STATISTICAL FINDINGS
# 

"""
1. Exploratory Data Analysis (EDA) Findings
-------------------------------------------
- Seasonal Demand Patterns:
  * Fall (Season 3) experiences peak demand with an average of ~234 rentals/hr.
  * Summer (Season 2) follows closely with ~215 rentals/hr.
  * Spring (Season 1) shows significantly lower demand (~116 rentals/hr), nearly 
    half of peak usage.
- Weather Sensitivity:
  * Clear / Partly Cloudy conditions (Weather 1) dominate overall usage (>66% of data) 
    and yield the highest rental volumes (~205 rentals/hr).
  * Adverse weather like Light Rain/Snow (Weather 3) severely drops demand to ~119 rentals/hr.
- Working Day vs. Non-Working Day Volume:
  * Overall average daily demand is very similar between working days (~193 rentals/hr) 
    and non-working days (~188 rentals/hr).
- Environmental Correlations:
  * Temperature (temp) and Feels-like temperature (atemp) show strong positive 
    correlations with rental count up to high temperatures.
  * Relative humidity (>65%) negatively impacts rental volume.


2. Hypothesis Testing Summary
-----------------------------
A. 2-Sample T-Test (Working Day Effect on Demand)
   - T-statistic: 1.2096 | P-value: 0.2264 (alpha = 0.05)
   - Decision: Fail to reject Null Hypothesis (H0).
   - Finding: There is no statistically significant difference in mean rental count 
     between working days and non-working days.

B. One-Way ANOVA (Weather Effect on Demand)
   - F-statistic: 65.5302 | P-value: 5.4821e-42 (alpha = 0.05)
   - Decision: Reject Null Hypothesis (H0).
   - Finding: Weather conditions have a statistically significant effect on bike rentals.

C. One-Way ANOVA (Season Effect on Demand)
   - F-statistic: 188.7292 | P-value: 6.1648e-118 (alpha = 0.05)
   - Decision: Reject Null Hypothesis (H0).
   - Finding: Season has a statistically significant effect on bike rental volume.

D. Chi-Square Test of Independence (Weather vs. Season Dependency)
   - Chi2 Statistic: 49.1586 | Degrees of Freedom: 9 | P-value: 1.549e-07 (alpha = 0.05)
   - Decision: Reject Null Hypothesis (H0).
   - Finding: Weather patterns and seasons are statistically dependent.
"""

print("Key Insights successfully documented as markdown/comments.")

# ==============================================================================
# STAGE 9: ACTIONABLE BUSINESS RECOMMENDATIONS FOR YULU
# ==============================================================================

"""
1. Fleet Management & Operational Optimization (Driven by Season & Weather ANOVA Insights)
-----------------------------------------------------------------------------------------
- Dynamic Fleet Reallocation: 
  Scale up active fleet deployment during peak demand periods (Fall/Season 3 & Summer/Season 2) 
  by ~30-40%. Downscale active bikes during Spring (Season 1) to reduce operational wear and tear.
- Weather-Responsive Maintenance Schedules: 
  Schedule major fleet overhauls, battery replacements, and hardware upgrades during Spring 
  and rainy/misty weather windows when natural user demand is lower.

2. Product & Equipment Enhancements (Driven by Weather ANOVA & Correlation Analysis)
------------------------------------------------------------------------------------
- All-Weather Vehicle Enhancements: 
  Introduce anti-slip tires, rain-resistant seat covers, and improved braking for wet conditions 
  to boost user safety and confidence during Light Rain/Mist (Weather 2 & 3).
- High-Humidity Protections: 
  Ensure battery enclosures and electronics carry enhanced IP-ratings to prevent hardware 
  degradation during humid conditions (>65% relative humidity).

3. Targeted Pricing & Incentive Strategies (Driven by 2-Sample T-Test & Weather Insights)
-----------------------------------------------------------------------------------------
- Segmented Fare Plans (Commuter vs. Leisure): 
  Though daily mean volume is equal across working and non-working days, usage behavior shifts. 
  Implement subscription-based "Commuter Passes" for peak rush hours on working days, and offer 
  discounted "Weekend/Leisure Passes" to maintain high bike utilization on non-working days.
- "Rainy Day" / Adverse Weather Discounts: 
  Offer automated surge discounts or bonus rewards when riding in Light Rain/Mist to incentivize 
  riders who would otherwise switch to alternative transport modes.

4. Demand Forecasting & Dynamic Operations (Driven by Chi-Square Test Results)
-------------------------------------------------------------------------------
- Interaction-Aware Predictive Models: 
  Because Weather and Season are statistically dependent (Chi-Square p < 0.05), machine learning 
  forecasting models must use feature interaction terms (e.g., Season x Weather) rather than treating 
  them as independent predictors.
- Location-Based Rebalancing: 
  On working days, focus rebalancing vans around metro stations, tech parks, and commercial hubs; 
  on non-working days, shift fleet concentration toward parks, tourist spots, and recreational areas.
"""

print("Business Recommendations successfully documented.")